In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from pathlib import Path
import joblib

In [2]:
digits = load_digits()

X = digits.data
y = digits.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("类别:", set(y))

X shape: (1797, 64)
y shape: (1797,)
类别: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)}


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("训练集:", X_train.shape)
print("测试集:", X_test.shape)

训练集: (1437, 64)
测试集: (360, 64)


In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([1437, 64])
torch.Size([1437])


In [6]:
class DigitsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        return self.net(x)

In [7]:
def train_model(model, optimizer, criterion, epochs=100):
    for epoch in range(epochs):
        model.train()
        
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 20 == 0:
            model.eval()
            
            with torch.no_grad():
                test_outputs = model(X_test_tensor)
                test_pred = torch.argmax(test_outputs, dim=1)
                test_acc = (test_pred == y_test_tensor).float().mean().item()
            
            print(
                f"Epoch [{epoch+1}/{epochs}], "
                f"Loss: {loss.item():.4f}, "
                f"Test Acc: {test_acc:.4f}"
            )

In [8]:
torch.manual_seed(42)

model = DigitsMLP()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    epochs=100
)

Epoch [20/100], Loss: 1.6299, Test Acc: 0.7500
Epoch [40/100], Loss: 0.6770, Test Acc: 0.8611
Epoch [60/100], Loss: 0.2546, Test Acc: 0.9306
Epoch [80/100], Loss: 0.1239, Test Acc: 0.9500
Epoch [100/100], Loss: 0.0711, Test Acc: 0.9639


In [9]:
model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_pred = torch.argmax(test_outputs, dim=1)
    test_acc = (test_pred == y_test_tensor).float().mean().item()

print("测试集准确率:", test_acc)

测试集准确率: 0.9638888835906982


In [10]:
save_dir = Path("saved_models")
save_dir.mkdir(exist_ok=True)

In [11]:
model_path = save_dir / "digits_mlp_state_dict.pth"

torch.save(model.state_dict(), model_path)

print("模型已保存到:", model_path)

模型已保存到: saved_models/digits_mlp_state_dict.pth


In [12]:
state_dict = model.state_dict()

for name, param in state_dict.items():
    print(name, param.shape)

net.0.weight torch.Size([128, 64])
net.0.bias torch.Size([128])
net.2.weight torch.Size([64, 128])
net.2.bias torch.Size([64])
net.4.weight torch.Size([10, 64])
net.4.bias torch.Size([10])


In [13]:
loaded_model = DigitsMLP()

loaded_model.load_state_dict(
    torch.load(model_path, weights_only=True)
)

loaded_model.eval()

print("模型加载完成")

模型加载完成


In [14]:
with torch.no_grad():
    loaded_outputs = loaded_model(X_test_tensor)
    loaded_pred = torch.argmax(loaded_outputs, dim=1)
    loaded_acc = (loaded_pred == y_test_tensor).float().mean().item()

print("加载后的模型测试准确率:", loaded_acc)

加载后的模型测试准确率: 0.9638888835906982


In [15]:
with torch.no_grad():
    original_outputs = model(X_test_tensor)
    loaded_outputs = loaded_model(X_test_tensor)

    original_pred = torch.argmax(original_outputs, dim=1)
    loaded_pred = torch.argmax(loaded_outputs, dim=1)

same = (original_pred == loaded_pred).float().mean().item()

print("两个模型预测结果一致比例:", same)

两个模型预测结果一致比例: 1.0


In [16]:
sample = X_test_tensor[0]

print(sample.shape)

torch.Size([64])


In [18]:
sample = sample.unsqueeze(0)

print(sample.shape)

torch.Size([1, 64])


In [19]:
loaded_model.eval()

with torch.no_grad():
    output = loaded_model(sample)
    pred = torch.argmax(output, dim=1)

print("模型预测结果:", pred.item())
print("真实标签:", y_test_tensor[0].item())

模型预测结果: 5
真实标签: 5


In [20]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
scaler_path = save_dir / "digits_scaler.pkl"

joblib.dump(scaler, scaler_path)

print("标准化器已保存到:", scaler_path)

标准化器已保存到: saved_models/digits_scaler.pkl


In [22]:
loaded_scaler = joblib.load(scaler_path)

print("标准化器加载完成")

标准化器加载完成


In [23]:
new_sample = digits.data[0].reshape(1, -1)

new_sample_scaled = loaded_scaler.transform(new_sample)

new_sample_tensor = torch.tensor(new_sample_scaled, dtype=torch.float32)

loaded_model.eval()

with torch.no_grad():
    output = loaded_model(new_sample_tensor)
    pred = torch.argmax(output, dim=1)

print("预测结果:", pred.item())
print("真实标签:", digits.target[0])

预测结果: 0
真实标签: 0


In [24]:
checkpoint_path = save_dir / "digits_mlp_checkpoint.pth"

checkpoint = {
    "epoch": 100,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": 0.0
}

torch.save(checkpoint, checkpoint_path)

print("checkpoint 已保存到:", checkpoint_path)

checkpoint 已保存到: saved_models/digits_mlp_checkpoint.pth


In [25]:
new_model = DigitsMLP()
new_optimizer = optim.Adam(new_model.parameters(), lr=0.001)

checkpoint = torch.load(checkpoint_path, weights_only=True)

new_model.load_state_dict(checkpoint["model_state_dict"])
new_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"]

print("从第", start_epoch, "轮继续训练")

从第 100 轮继续训练


In [31]:
train_model(
    model=new_model,
    optimizer=new_optimizer,
    criterion=criterion,
    epochs=50
)

Epoch [20/50], Loss: 0.0026, Test Acc: 0.9694
Epoch [40/50], Loss: 0.0023, Test Acc: 0.9694
